In [116]:
import numpy as np
from numpy.typing import NDArray

from toporisk.topology import Diagram

# A sampled landscape: shape (num_layers, resolution). Every landscape in a
# comparison MUST share the same (x_min, x_max, resolution) so they can be
# averaged pointwise.
Landscape = NDArray[np.float64]

In [3]:
arr =  np.random.rand(10, 2)

In [139]:
diagram = np.array([
    [0.0, 0.7],
    [0.1, 0.5],
    [0.3, 1.1],
    [0.8, 1.6],
])

In [ ]:
def persistence_landscape(
    diagram: Diagram,
    num_layers: int = 1,
    resolution: int = 500,
    x_range: tuple[float, float] | None = None
) -> Landscape:
    if diagram is None or diagram.size == 0:
            return np.zeros((num_layers, resolution))
    if x_range is None:
        x_range = (0.0, float(diagram[:, 1].max()))
    tent_values_at_t = np.zeros((diagram.shape[0], resolution))
    for i in range(diagram.shape[0]):
        birth = diagram[i, 0]
        death = diagram[i, 1]
        if birth > death:
            raise ValueError(f"Invalid diagram point: {diagram[i]} (birth > death)")
        if birth < 0 or death > x_range[1]:
            raise ValueError(f"Diagram point {diagram[i]} is outside the x_range {x_range}")
       
       #partitions interval x_range into ticks divided equally by 'resolution' argument and calculates
       # tent function values at each tick for the current diagram point i. Saves the values in tent_values_at_t array.
        for t in range(resolution):
            tick = x_range[0] + t * (x_range[1] - x_range[0]) / (resolution - 1)
            tent_values_at_t[i, t]  = max(0, min(tick - birth, death - tick))


    eta = np.zeros((num_layers, resolution))
    for t in range(resolution):
        sorted_tent_values = np.sort(tent_values_at_t[:, t])[::-1]
        for k in range(num_layers):
            if k < len(sorted_tent_values):
                eta[k, t] = sorted_tent_values[k]
            else:
                eta[k, t] = 0.0

    return eta
    """Sampled persistence landscape of a (finite) diagram.  [TODO]

    eta_k (t) = k-th largest value of Lambda_{(b,d)}(t) over all points in the diagram.
    num_layers = number of landscape layers eta_1, ..., eta_{num_layers} to sample (default is 1 as per paper).

    The landscape turns each finite pair (b, d) into a tent function

        Lambda_{(b,d)}(t) = max( 0, min(t - b, d - t) ),

    and the k-th landscape layer lambda_k(t) is the k-th largest tent value at t.
    Sample lambda_1..lambda_{num_layers} on a grid of ``resolution`` points over
    ``x_range`` and return them stacked as shape ``(num_layers, resolution)``.

    Paper values: ``num_layers = 1`` (k = 1) is what Goel-Sharma-Kanniainen use.

    Critical design constraint: ``x_range`` must be *fixed* (passed in, shared
    across all sub-windows of an asset), NOT inferred per-diagram. Otherwise two
    landscapes live on different grids and cannot be averaged pointwise, which
    breaks the mean-landscape step. If ``x_range`` is None, decide on a sensible
    shared default and document it -- but the caller in ``risk.py`` should pin it.

    You may build the tent functions yourself (good for understanding), or use
    ``gudhi.representations.Landscape`` with a fixed ``sample_range``. Either way,
    handle the empty diagram: return zeros of shape ``(num_layers, resolution)``.

    Args:
        diagram: A finite H0 diagram, ``(n, 2)``.
        num_layers: Number of landscape layers to sample.
        resolution: Number of grid points.
        x_range: ``(x_min, x_max)`` sampling range; must be shared across a
            comparison set.

    Returns:
        Array of shape ``(num_layers, resolution)``.
    """

    raise NotImplementedError("implement persistence_landscape")

In [ ]:
persistence_landscape(diagram, num_layers=1, resolution=500, x_range=(0, 2))

array([0.35135159, 0.68049641, 0.82851276, 0.38627813, 0.83098369,
       0.37560502, 0.82520815, 0.36469229, 0.95395828, 0.19024661,
       0.05434733, 0.51109196, 0.60134771, 0.478738  , 0.10071686,
       0.21506914, 0.25666785, 0.29470091, 0.42305878, 0.36811802,
       0.11736857, 0.08052435, 0.8420872 , 0.49925013, 0.91158608,
       0.67449151, 0.77501922, 0.76175101, 0.41618624, 0.26587868,
       0.06148239, 0.5606884 , 0.33300264, 0.62166225, 0.89928886,
       0.93344389, 0.25716907, 0.26488279, 0.16381369, 0.66530017,
       0.39042158, 0.51141767, 0.98002888, 0.42439532, 0.45857028,
       0.69556794, 0.36373469, 0.30458562, 0.46610829, 0.42948848,
       0.36188273, 0.20712321, 0.84794525, 0.60190283, 0.35522011,
       0.46603566, 0.68365478, 0.30758415, 0.85072615, 0.21147929,
       0.53463568, 0.13819449, 0.27309644, 0.02692285, 0.96271069,
       0.28875185, 0.0730447 , 0.05519473, 0.89827299, 0.48030538,
       0.28863762, 0.48466239, 0.61476898, 0.23779087, 0.15695

In [100]:

def takens_embedding(
    series: NDArray[np.float64],
    d: int = 3,
    tau: int = 1,
) -> NDArray[np.float64]:
    """Time-delay (Takens) embedding of a 1-D series into R^d.  [TODO]

    Map the scalar series ``s`` to a cloud of delay vectors. Row ``j`` is

        ( s[j], s[j + tau], s[j + 2*tau], ..., s[j + (d-1)*tau] ).

    The number of rows is ``len(series) - (d - 1) * tau`` and each row lives in
    R^d, so the output has shape ``(len(series) - (d-1)*tau, d)``.

    Paper values: ``d = 3``, ``tau = 1`` (fixed by convention, not tuned; the
    authors flag parameter selection as future work).

    Guard: raise ``ValueError`` if the series is too short to form even one
    delay vector, i.e. if ``len(series) - (d-1)*tau < 1``.

    Args:
        series: 1-D series (one asset's returns over one sub-window).
        d: Embedding dimension.
        tau: Time delay.

    Returns:
        Array of shape ``(len(series) - (d-1)*tau, d)``.

    Raises:
        ValueError: If the series is too short, or d/tau are non-positive.
    """
    if d <= 0 or tau <= 0:
        raise ValueError("Embedding dimension and time delay must be positive.")

    if len(series) - (d - 1) * tau < 1:
        raise ValueError("Series is too short to form even one delay vector.")
    n_rows = len(series) - (d - 1) * tau
    
    # Create a 2D matrix of indices using broadcasting
    # row_indices shape: (n_rows, 1)
    # col_offsets shape: (1, d)
    row_indices = np.arange(n_rows)[:, None]
    col_offsets = np.arange(d)[None, :] * tau
    
    # Advanced integer indexing creates the complete (n_rows, d) array instantly
    return series[row_indices + col_offsets]
    


In [101]:
takens_embedding(arr, d=4, tau=1)

array([[0.35135159, 0.68049641, 0.82851276, 0.38627813],
       [0.68049641, 0.82851276, 0.38627813, 0.83098369],
       [0.82851276, 0.38627813, 0.83098369, 0.37560502],
       [0.38627813, 0.83098369, 0.37560502, 0.82520815],
       [0.83098369, 0.37560502, 0.82520815, 0.36469229],
       [0.37560502, 0.82520815, 0.36469229, 0.95395828],
       [0.82520815, 0.36469229, 0.95395828, 0.19024661],
       [0.36469229, 0.95395828, 0.19024661, 0.05434733],
       [0.95395828, 0.19024661, 0.05434733, 0.51109196],
       [0.19024661, 0.05434733, 0.51109196, 0.60134771],
       [0.05434733, 0.51109196, 0.60134771, 0.478738  ],
       [0.51109196, 0.60134771, 0.478738  , 0.10071686],
       [0.60134771, 0.478738  , 0.10071686, 0.21506914],
       [0.478738  , 0.10071686, 0.21506914, 0.25666785],
       [0.10071686, 0.21506914, 0.25666785, 0.29470091],
       [0.21506914, 0.25666785, 0.29470091, 0.42305878],
       [0.25666785, 0.29470091, 0.42305878, 0.36811802],
       [0.29470091, 0.42305878,

In [102]:
  row_indices = np.arange(10)[:, None]

In [107]:
row_indices

array([[0],
       [1],
       [2],
       [3],
       [4],
       [5],
       [6],
       [7],
       [8],
       [9]])

In [109]:
 col_offsets = np.arange(3)[None, :] * 2

In [111]:
col_offsets

array([[0, 2, 4]])

In [113]:
series = np.array(row_indices + col_offsets)

In [2]:
series

NameError: name 'series' is not defined

In [28]:
arr=np.random.rand(300)

In [ ]:
"""Asset topological risk and the risk matrix Q.

``asset_topological_risk`` is the capstone: it composes every earlier piece and
is where the norm-of-mean subtlety actually bites. It is a stub. ``risk_matrix``
is trivial glue.
"""

from __future__ import annotations

import numpy as np
from numpy.typing import NDArray

from toporisk.embedding import (  # noqa: F401  (used when you implement)
    sub_windows,
    takens_embedding,
)
from toporisk.landscape import lp_norm, mean_landscape, persistence_landscape  # noqa: F401
from toporisk.topology import finite_pairs, persistence_diagram_h0  # noqa: F401


def asset_topological_risk(
    series: NDArray[np.float64],
    sub_len: int = 126,
    shift: int = 21,
    d: int = 3,
    tau: int = 1,
    p: float = 1.0,
    num_layers: int = 1,
    resolution: int = 500,
) -> float:
    if len(series) < sub_len:
        raise ValueError(f"series length {len(series)} too short for sub_len {sub_len}")
    global_max = 0.0
    landscape = []
    finite_diagrams = []
    for sub_window in sub_windows(series, sub_len, shift):
        point_cloud = takens_embedding(sub_window, d=d, tau=tau)
        diagram = persistence_diagram_h0(point_cloud)
        finite_diagrams.append(finite_pairs(diagram))
    deaths = np.concatenate(finite_diagrams)[:, 1]
    if deaths.size == 0: 
        return 0.0
    
    global_max = deaths.max()

    for finite_diagram in finite_diagrams:
        landscape.append(persistence_landscape(finite_diagram, num_layers=num_layers, resolution=resolution, x_range=(0.0, global_max)))
    
    eta_bar = mean_landscape(landscape)
    Lambda = sum((lp_norm(eta, p) - lp_norm(eta_bar, p))**2 for eta in landscape)
    return Lambda


In [32]:
asset_topological_risk(arr, sub_len=126, shift=21, d=3, tau=1, p=1.0, num_layers=1, resolution=500)

diagram 0
diagram 1
diagram 2
diagram 3
diagram 4
diagram 5
diagram 6
diagram 7
diagram 8


546.6288466954412

In [14]:
arr=[]
if not arr:
    raise ValueError(f"series length {len(arr)} too short for sub_len {126}")

ValueError: series length 0 too short for sub_len 126